# YOLO11n-seg + SimAM Attention before Segment Head

Experiment:
- Baseline: YOLO11n-seg
- Modified: YOLO11n-seg + SimAM before Segment Head
- Task: shrimp disease instance segmentation


## Colab Quick Setup

Run this cell first on Google Colab. Upload or mount the full `yolov11n_attention` folder, then edit `COLAB_WORK_DIR` if your folder is not in one of the default locations.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IS_COLAB = "google.colab" in sys.modules
COLAB_WORK_DIR = None  # Example: "/content/drive/MyDrive/yolov11n_attention"
GITHUB_REPO_URL = "https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git"
GITHUB_SUBDIR = "yolov11n_attention"

print("Running in Colab:", IS_COLAB)

if IS_COLAB:
    # Optional Google Drive mount:
    # from google.colab import drive
    # drive.mount('/content/drive')

    candidates = []
    if COLAB_WORK_DIR:
        candidates.append(Path(COLAB_WORK_DIR))
    candidates.extend([
        Path.cwd(),
        Path("/content/yolov11n_attention"),
        Path("/content/drive/MyDrive/yolov11n_attention"),
    ])

    work_dir = next(
        (
            p
            for p in candidates
            if (p / "yolo11n_seg_simam_experiment.ipynb").exists()
            or (p / "setup_simam_ultralytics.py").exists()
            or (p / "ultralytics").exists()
        ),
        None,
    )
    if work_dir is None and GITHUB_REPO_URL:
        repo_dir = Path("/content") / Path(GITHUB_REPO_URL).stem
        if not repo_dir.exists():
            subprocess.run(["git", "clone", GITHUB_REPO_URL, str(repo_dir)], check=True)
        work_dir = repo_dir / GITHUB_SUBDIR
    if work_dir is None:
        raise FileNotFoundError(
            "Could not find yolov11n_attention. Upload/clone the folder or set COLAB_WORK_DIR."
        )
    if not work_dir.exists():
        raise FileNotFoundError(f"Configured work_dir does not exist: {work_dir}")

    os.chdir(work_dir)
    print("Working directory:", Path.cwd())
    setup_script = Path("setup_simam_ultralytics.py")
    if setup_script.exists():
        subprocess.run([sys.executable, str(setup_script)], check=True)
    elif not (Path("ultralytics") / "ultralytics").exists():
        subprocess.run(["git", "clone", "https://github.com/ultralytics/ultralytics.git", "ultralytics"], check=True)
        print("WARNING: setup_simam_ultralytics.py was not found, so SimAM patches were not applied.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "ultralytics"], check=True)
    print("Editable Ultralytics install complete.")
else:
    print("Not running in Colab. Continue with the local environment cells below.")


In [ ]:
import os
import sys
from pathlib import Path
import torch

ROOT = Path.cwd()
print("Current working directory:", ROOT)
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Dataset Setup from Baseline Notebook

This uses the same Roboflow dataset and split logic as `baseline-sweep-yolov11m.ipynb`: workspace `shirmpdiseasedtection`, project `ewu_shrimp_disease`, version `1`, format `yolo26`, seed `42`.


In [ ]:
import importlib.util
import os
import random
import shutil
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("roboflow") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow"])

import yaml
from roboflow import Roboflow

SEED = 42
random.seed(SEED)

ROBOFLOW_API_KEY_DIRECT = ""  # Do not commit secrets. Use Colab Secret or environment variable instead.
ROBOFLOW_WORKSPACE = "shirmpdiseasedtection"
ROBOFLOW_PROJECT = "ewu_shrimp_disease"
ROBOFLOW_VERSION = 1
ROBOFLOW_FORMAT = "yolo26"


def get_roboflow_api_key():
    if ROBOFLOW_API_KEY_DIRECT.strip():
        return ROBOFLOW_API_KEY_DIRECT.strip()
    try:
        from google.colab import userdata

        key = userdata.get("ROBOFLOW_API_KEY")
        if key:
            return key.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key:
            return key.strip()
    except Exception:
        pass
    return os.environ.get("ROBOFLOW_API_KEY", "").strip()


if "google.colab" in sys.modules:
    DOWNLOAD_ROOT = Path("/content")
elif Path("/kaggle/working").exists():
    DOWNLOAD_ROOT = Path("/kaggle/working")
else:
    DOWNLOAD_ROOT = Path.cwd()

DATASET_DIR = DOWNLOAD_ROOT / "ewu_shrimp_disease-1"
DATA_YAML = str(DATASET_DIR / "data.yaml")
TEST_SOURCE = str(DATASET_DIR / "test" / "images")
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

if not Path(DATA_YAML).exists():
    api_key = get_roboflow_api_key()
    if not api_key:
        raise RuntimeError(
            "Missing Roboflow API key. In Colab, add a secret named ROBOFLOW_API_KEY, "
            "or set os.environ['ROBOFLOW_API_KEY'] before running this cell."
        )

    rf = Roboflow(api_key=api_key)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    version = project.version(ROBOFLOW_VERSION)
    dataset = version.download(ROBOFLOW_FORMAT, location=str(DATASET_DIR))
    DATASET_DIR = Path(dataset.location)
    DATA_YAML = str(DATASET_DIR / "data.yaml")
    TEST_SOURCE = str(DATASET_DIR / "test" / "images")
else:
    print("Existing Roboflow dataset found:", DATASET_DIR)

train_path = DATASET_DIR / "train"
for split in ["valid", "test"]:
    for sub in ["images", "labels"]:
        (DATASET_DIR / split / sub).mkdir(parents=True, exist_ok=True)

valid_images_dir = DATASET_DIR / "valid" / "images"
test_images_dir = DATASET_DIR / "test" / "images"

if not any(valid_images_dir.glob("*")) and not any(test_images_dir.glob("*")):
    train_images_dir = train_path / "images"
    image_files = [f.name for f in train_images_dir.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
    if not image_files:
        raise FileNotFoundError(f"No training images found in {train_images_dir}")

    random.shuffle(image_files)
    train_count = int(0.8 * len(image_files))
    val_count = int(0.1 * len(image_files))
    val_files = image_files[train_count:train_count + val_count]
    test_files = image_files[train_count + val_count:]

    def move_files(files, target_split):
        for f in files:
            shutil.move(str(train_path / "images" / f), str(DATASET_DIR / target_split / "images" / f))
            label_f = f.rsplit(".", 1)[0] + ".txt"
            label_src = train_path / "labels" / label_f
            label_dst = DATASET_DIR / target_split / "labels" / label_f
            if label_src.exists():
                shutil.move(str(label_src), str(label_dst))
            else:
                label_dst.write_text("")

    move_files(val_files, "valid")
    move_files(test_files, "test")
    print(f"Split complete: {len(image_files) - len(val_files) - len(test_files)} train, {len(val_files)} val, {len(test_files)} test")
else:
    print("Existing valid/test split detected. Keeping downloaded split.")

with open(DATA_YAML, "r") as f:
    data_config = yaml.safe_load(f)

data_config["train"] = str(DATASET_DIR / "train" / "images")
data_config["val"] = str(DATASET_DIR / "valid" / "images")
data_config["test"] = str(DATASET_DIR / "test" / "images")

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print("DATASET_DIR:", DATASET_DIR)
print("DATA_YAML:", DATA_YAML)
print("TEST_SOURCE:", TEST_SOURCE)
print("Classes:", data_config.get("names", []))


In [ ]:
from pathlib import Path

WORK_DIR = Path.cwd()
ULTRALYTICS_DIR = WORK_DIR / "ultralytics"
MODEL_YAML = ULTRALYTICS_DIR / "ultralytics/cfg/models/11/yolo11n-seg-simam-head.yaml"

DATA_YAML = globals().get("DATA_YAML", "/path/to/your/data.yaml")
TEST_SOURCE = globals().get("TEST_SOURCE", "/path/to/test/images")

PROJECT_DIR = "runs/shrimp_yolo11n_seg_simam"
EXP_NAME = "yolo11n_seg_simam_head"

IMG_SIZE = 640
EPOCHS = 100
BATCH = 8
DEVICE = 0
WORKERS = 4

print("WORK_DIR:", WORK_DIR)
print("ULTRALYTICS_DIR:", ULTRALYTICS_DIR)
print("MODEL_YAML:", MODEL_YAML)
print("DATA_YAML:", DATA_YAML)
print("TEST_SOURCE:", TEST_SOURCE)


In [ ]:
from pathlib import Path

required_files = {
    "Ultralytics repo": ULTRALYTICS_DIR,
    "SimAM model YAML": MODEL_YAML,
    "data.yaml": Path(DATA_YAML),
}

for label, path in required_files.items():
    if path.exists():
        print(f"OK: {label}: {path}")
    else:
        print(f"WARNING: {label} not found: {path}")

if not ULTRALYTICS_DIR.exists():
    print("Fix: run this notebook from the yolov11n_attention directory or clone Ultralytics into ./ultralytics.")
if not MODEL_YAML.exists():
    print("Fix: confirm ultralytics/cfg/models/11/yolo11n-seg-simam-head.yaml exists in the local source tree.")
if not Path(DATA_YAML).exists():
    print("Fix: edit DATA_YAML in the config cell so it points to your dataset data.yaml before training.")


In [ ]:
import sys
sys.path.insert(0, str(ULTRALYTICS_DIR))

from ultralytics import YOLO
import ultralytics

print("Ultralytics version:", ultralytics.__version__)
print("Ultralytics source:", ultralytics.__file__)


In [ ]:
from ultralytics import YOLO

try:
    model = YOLO(str(MODEL_YAML))
    model.info(verbose=True)
    print("Model YAML build successful.")
except Exception as e:
    print("Model YAML build failed:")
    print(e)
    print("Check these items:")
    print("- SimAM is imported in ultralytics/nn/modules/__init__.py")
    print("- SimAM is imported in ultralytics/nn/tasks.py")
    print("- parse_model() handles SimAM as a channel-preserving module")
    print("- YAML layer indexes are correct")
    print("- Segment receives the three SimAM output layers")
    raise


In [ ]:
import torch

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

try:
    model.model.eval()
    with torch.no_grad():
        out = model.model(dummy)
    print("Forward test successful.")
    print(type(out))
except Exception as e:
    print("Forward test failed:")
    print(e)
    raise


In [ ]:
model = YOLO(str(MODEL_YAML))

try:
    model.load("yolo11n-seg.pt")
    print("Loaded pretrained yolo11n-seg.pt successfully.")
except Exception as e:
    print("Could not load pretrained weights completely.")
    print("This can happen because the architecture was modified.")
    print(e)


In [ ]:
if not Path(DATA_YAML).exists():
    print(f"Training skipped because DATA_YAML does not exist: {DATA_YAML}")
    print("Edit DATA_YAML in the config cell, then rerun this cell.")
    results = None
else:
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        project=PROJECT_DIR,
        name=EXP_NAME,
        pretrained=True,
        optimizer="AdamW",
        lr0=0.001,
        cos_lr=True,
        patience=30,
        close_mosaic=10,
        mask_ratio=4,
        overlap_mask=True,
    )


In [ ]:
BEST_PT = Path(PROJECT_DIR) / EXP_NAME / "weights" / "best.pt"
print("BEST_PT:", BEST_PT)

if not BEST_PT.exists():
    print("Validation skipped because best.pt was not found. Train first or update BEST_PT.")
elif not Path(DATA_YAML).exists():
    print(f"Validation skipped because DATA_YAML does not exist: {DATA_YAML}")
else:
    best_model = YOLO(str(BEST_PT))

    metrics = best_model.val(
        data=DATA_YAML,
        imgsz=IMG_SIZE,
        device=DEVICE,
        split="val",
    )

    print("Box mAP50:", metrics.box.map50)
    print("Box mAP50-95:", metrics.box.map)

    if hasattr(metrics, "seg"):
        print("Mask mAP50:", metrics.seg.map50)
        print("Mask mAP50-95:", metrics.seg.map)
    else:
        print("No segmentation metrics found.")


In [ ]:
TEST_SOURCE = globals().get("TEST_SOURCE", "/path/to/test/images")

if "best_model" not in globals():
    print("Prediction skipped because best_model is not loaded. Run validation or load BEST_PT first.")
elif not Path(TEST_SOURCE).exists():
    print(f"Prediction skipped because TEST_SOURCE does not exist: {TEST_SOURCE}")
    print("Edit TEST_SOURCE to a test image, directory, or video path.")
else:
    pred_results = best_model.predict(
        source=TEST_SOURCE,
        imgsz=IMG_SIZE,
        conf=0.25,
        iou=0.5,
        save=True,
        save_txt=True,
        save_conf=True,
        project="runs/predict_shrimp_simam",
        name="simam_predict",
    )


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

predict_dir = Path("runs/predict_shrimp_simam/simam_predict")
image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = sorted([p for p in predict_dir.rglob("*") if p.suffix.lower() in image_exts]) if predict_dir.exists() else []

if not images:
    print(f"No prediction images found in {predict_dir}. Run the predict cell first.")
else:
    show_images = images[:8]
    cols = min(4, len(show_images))
    rows = (len(show_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if len(show_images) == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for ax, img_path in zip(axes, show_images):
        ax.imshow(Image.open(img_path))
        ax.set_title(img_path.name)
        ax.axis("off")

    for ax in axes[len(show_images):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
if "best_model" not in globals():
    print("ONNX export skipped because best_model is not loaded.")
else:
    best_model.export(
        format="onnx",
        imgsz=IMG_SIZE,
        simplify=True,
        opset=12,
    )


In [ ]:
if "best_model" not in globals():
    print("TFLite export skipped because best_model is not loaded.")
else:
    try:
        best_model.export(
            format="tflite",
            imgsz=IMG_SIZE,
            half=True,
        )
    except Exception as e:
        print("TFLite export failed. This may require TensorFlow / onnx2tf dependencies.")
        print(e)


## Experiment Tracking

| Model | Mask mAP50 | Mask mAP50-95 | Box mAP50 | Params | FLOPs | Latency | FPS | Size |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| YOLO11n-seg baseline | | | | | | | | |
| YOLO11n-seg + SimAM Head | | | | | | | | |


## Common Errors and Fixes

| Error | Likely cause | Fix |
|---|---|---|
| `KeyError: 'SimAM'` | `SimAM` is not imported into `tasks.py` globals | Confirm `from ultralytics.nn.modules import SimAM` exists in `ultralytics/nn/tasks.py`. |
| `NameError` or import error for `SimAM` | `SimAM` is not exported from `ultralytics/nn/modules/__init__.py` | Confirm `SimAM` is imported from `.conv` and included in `__all__`. |
| Shape mismatch at `Segment` | The final `Segment` layer does not point to the three SimAM outputs | Confirm the YAML ends with SimAM layers `23, 24, 25` and `Segment` uses `[[23, 24, 25], 1, Segment, ...]`. |
| Dataset file not found | `DATA_YAML` still points to the placeholder path | Edit `DATA_YAML` in the config cell. |
| TFLite export fails | TensorFlow, onnx2tf, or related export dependencies are missing | Install the required export dependencies or use ONNX export first. |
